# Setup

## Imports

In [ ]:
import numpy as np
import pandas as pd

from typing import Optional

from rlbench.action_modes.action_mode import MoveArmThenGripper
from rlbench.action_modes.arm_action_modes import JointVelocity
from rlbench.action_modes.gripper_action_modes import Discrete

from rlbench.environment import Environment
from rlbench.task_environment import TaskEnvironment
from rlbench.backend.task import Task
from rlbench.demo import Demo
from rlbench.observation_config import ObservationConfig, CameraConfig


from pyrep.const import RenderMode
from pyrep.objects import Object

import numpy as np
import torch
import torch.nn.functional as F
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from lib.cam_type import CamType
from lib.agent import Agent
from lib.policy_type import PolicyType
from lib.utils import save_demos, load_demos



from seed import set_seed
from lib.utils import get_task_name
from task_utils import run_grasp_with_agent, demos_and_train_for_task, run_determined_grasp_with_agent

from itertools import product


In [ ]:
## Vision Experiments - Grasp
from rlbench.tasks.vision_static import VisionStatic as Vision_Static
from rlbench.tasks.vision_random import VisionRandom as Vision_Random

## Environment Launch


In [ ]:
live_demos = True
DATASET = '' if live_demos else 'PATH/TO/YOUR/DATASET'

obs_config = ObservationConfig()
obs_config.set_all(True) ## important to get the data from the joints etc
cam_config = CameraConfig(
  rgb=True, 
  depth=True, 
  mask=True, 
  point_cloud=True,
  render_mode=RenderMode.OPENGL,
  image_size=(64, 64)
)
nocam_config = CameraConfig(rgb=False, depth=False, mask=False,
                          render_mode=RenderMode.OPENGL)

## active cameras:
obs_config.right_shoulder_camera = cam_config
obs_config.left_shoulder_camera = cam_config
obs_config.wrist_camera = cam_config

obs_config.overhead_camera = nocam_config
obs_config.front_camera = nocam_config


action_mode = MoveArmThenGripper(
    arm_action_mode=JointVelocity(), gripper_action_mode=Discrete())

env = Environment(
    action_mode, DATASET, obs_config, False)
env.launch()


## Helper Functions to be used later


In [ ]:
def train_and_test_vision_config(
  env: Environment,
  agents: list[Agent], 
  params: list[dict],
  train_demos: list[Demo],
  test_demos: list[Demo],
  task: type[Task],
  train_task_params: dict,
  test_task_params: dict,
  df_csv_name: str,
  check_random_demos: Optional[int] = None,
  save_individual: bool = False,
  skip_train: bool = False
) -> tuple[pd.DataFrame, dict]:
  
  df_cols = [
    "task_name",
    "agent_name",
    "demo_type", 
    "task_params",
    "demo_index",
    "is_success", 
    "final_distance",
    "min_distance",
    "max_eplen", 
    "gripper_image_paths"
  ]
  
  def run_helper(
    ags: list[Agent],
    task_params: dict,
    demos: list[Demo],
    demo_type: str,
    df_str: str,
    save: bool
  ) -> pd.DataFrame:
    
    frame = pd.DataFrame(columns=df_cols, index = range(len(demos) * len(ags)))
    offset = 0

    for agent in ags:
      task_env = env.get_task(task, **task_params)
      dicts = run_determined_grasp_with_agent(
        task_env,
        agent, 
        demos, 
        "demo_max"
      )
      for i in range(len(demos)):
        frame.loc[i + offset] = {
          "task_name": get_task_name(task),
          "agent_name": agent,
          "demo_type": demo_type,
          "task_params": task_params,
          "demo_index": i,
          "is_success": dicts[i]["done"],
          "final_distance": dicts[i]["distances"][-1],
          "min_distance": min(dicts[i]["distances"]),
          "max_eplen": dicts[i]["max_eplen"] ,
          "gripper_image_paths": dicts[i]["gripper_image_paths"]
        }
      offset += len(demos)
      
    if save: frame.to_csv(f"{df_str}.csv", index = True) 
    return frame
    

  dfs: list[pd.DataFrame] = []
  ret_dict = {}

  ## =================== TRAIN
  if not skip_train:
    for agent, param in zip(agents, params):
      print(f"Training Agent '{agent}'")
      demos_and_train_for_task(
        env,
        task, 
        agent,
        train_demos,
        training_params=param,
        task_params=train_task_params
      )
      print()

  # ==========================

  print(f"Testing agents on the trained demos (train_demos):")
  df_control = run_helper(
    agents,
    train_task_params,
    train_demos, 
    demo_type= "control",
    df_str = f"{df_csv_name}--control",
    save = save_individual
  )
  dfs.append(df_control)
  print()
  


  if check_random_demos is not None and isinstance(check_random_demos, int) and check_random_demos > 0:
    print(f"Checking on random {check_random_demos} demos withing the training spec, using (train_task_params)")
    task_env = env.get_task(task, **train_task_params)
    random_demos = task_env.get_demos(check_random_demos, live_demos = True)
    ret_dict["random_demos"] = random_demos

    ## Currently only uses train params in random task
    df_random = run_helper(
      agents,
      train_task_params,
      random_demos, 
      demo_type= "random",
      df_str = f"{df_csv_name}--random",
      save = save_individual
    )
    dfs.append(df_random)

    print()


  print(f"Testing agents on the testing demos")
  
  df_test = run_helper(
    agents ,
    test_task_params,
    test_demos, 
    demo_type = "test",
    df_str = f"{df_csv_name}--test",
    save = save_individual
  )
  dfs.append(df_test)
  print()


  all_df = pd.concat(dfs, axis = 0, ignore_index=True)
  all_df.to_csv(f"{df_csv_name}.csv", index = True)
  return all_df, ret_dict



# Shutdown


In [ ]:
env.shutdown()

# Grasping Task - Camera Experiments

Here I am investigating the use of multiple cameras aiding a more complex task like a grasping task


## (Static) Grasping with Single Camera
Following form earlier, I want to try each of the cameras I have with this task to see which performs better per static cube and a single demo.

In [ ]:
cam_types = [
  CamType.WRIST, ## TODO: wrist seems to perform better with longer epispdes do a run with that
  # CamType.LEFT_SHOULDER,
  # CamType.RIGHT_SHOULDER,
]

task = Vision_Static
## configure the task, so that it is always the same per test here
task_env = env.get_task(task, scale = 1, wrist_cam_distance = 0.5)

training_params = {
  "epochs": 1000,
  "minibatch_size": 64,
  "lr": 1e-3,
  "shuffle_obs_in_demo": False,
  "shuffle_data": False,
  # "lock_loader_seed": 42 ## TEST: test this later not sure
  # "lambda_grasp_loss": 20 ## 1 by default
}

task_params = {
  "scale" : 1.0, 
  "wrist_cam_distance": 0.5
}
max_eplens = ["demo_max", 150]
repeats = 20

In [ ]:
demos = task_env.get_demos(1, live_demos = True)


In [ ]:
df = pd.DataFrame(columns=[
  "task_name", "cam_type", "demo_count", "max_eplen",
  "is_success", "min_distance", "final_distance",
  "gripper_image_paths"
  
], index = range(repeats * len(cam_types) * 2)) 

In [ ]:
for i, (cam_type, max_eplen, rep) in enumerate(product(cam_types, max_eplens, range(repeats))):

  print(f"Running {get_task_name(task)} with {cam_type} and 1 demo, repeat: {rep}")

  agent = Agent(
    env.action_shape[0],
    PolicyType.SIMPLE_GRASP,
    cam_type,
    # grasp_thresh = 0.5, this is default
  ) 

  # max_eplen = 250 if cam_type == CamType.WRIST else "demo_max"


  rets, done = run_grasp_with_agent(
    env, task, agent, demos, max_eplen = max_eplen, ## TODO: try 'demo_max'?? 

    training_params=training_params, 
    task_params=task_params,
    
    print_index=i
  )

  df.loc[i] = {
    "task_name": get_task_name(task),
    "cam_type": cam_type,
    "demo_count": 1,
    "max_eplen": max_eplen,
    "is_success": done,
    "min_distance": min(rets["distances"]),
    "final_distance": rets["distances"][-1],
    "gripper_image_paths": rets["gripper_image_paths"]

  }
  df.to_csv("vision_single_static-just_wrist-results.csv", index=True)



## (Static) Grasping with combinations of cameras


In [ ]:
list(filter(lambda ct: ct not in cam_types, CamType.all_combinations()))

In [ ]:
cam_types_single = cam_types
## all cameras
cam_types = list(filter(lambda ct: ct not in cam_types_single, CamType.all_combinations()))

task = Vision_Static
## configure the task, so that it is always the same per test here
task_env = env.get_task(task, scale = 1, wrist_cam_distance = 0.5)

training_params = {
  "epochs": 1000,
  "minibatch_size": 64,
  "lr": 1e-3,
  "shuffle_obs_in_demo": False,
  "shuffle_data": False,
  # "lock_loader_seed": 42 ## TEST: test this later not sure
  # "lambda_grasp_loss": 20 ## 1 by default
}

task_params = {
  "scale" : 1.0, 
  "wrist_cam_distance": 0.5
}

max_eplens = ["demo_max", 150]

repeats = 20
cam_types

In [ ]:
df = pd.DataFrame(columns=[
  "task_name", "cam_type", "demo_count", "max_eplen",
  "is_success", "min_distance", "final_distance",
  "gripper_image_paths"
  
], index = range(repeats * len(cam_types) * 2))

In [ ]:
for i, (cam_type, max_eplen, rep) in enumerate(product(cam_types, max_eplens, range(repeats))):

  print(f"Running {get_task_name(task)} with {cam_type} and 1 demo, repeat: {rep}")

  agent = Agent(
    env.action_shape[0],
    PolicyType.SIMPLE_GRASP,
    cam_type,
    # grasp_thresh = 0.5, this is default
  ) 

  max_eplen = "demo_max"

  rets, done = run_grasp_with_agent(
    env, task, agent, demos, max_eplen = max_eplen, ## TODO: try 'demo_max'?? 

    training_params=training_params, 
    task_params=task_params,
    
    print_index=i
  )

  df.loc[i] = {
    "task_name": get_task_name(task),
    "cam_type": cam_type,
    "demo_count": 1,
    "max_eplen": max_eplen,
    "is_success": done,
    "min_distance": min(rets["distances"]),
    "final_distance": rets["distances"][-1],
    "gripper_image_paths": rets["gripper_image_paths"]

  }
  df.to_csv("vision_single_static-comb_cams-results.csv", index=True)



## Param Tuning - Trying longer sequences
See `plots.ipynb` for the explanation but larger mb size hence longer sequences of uninterrupted actions from the observations dataset will help the robot

TODO: Maybe change the dataset so that the robot trains on entirety of the given observations?

In [ ]:
cam_types = [
  CamType.WRIST, 
  CamType.RIGHT_SHOULDER,
  CamType.LEFT_SHOULDER,
  CamType.LEFT_SHOULDER | CamType.RIGHT_SHOULDER
]

task = Vision_Static
task_env = env.get_task(task, scale = 1, wrist_cam_distance = 0.5)

training_params = {
  "epochs": 1000,
  "minibatch_size": None, ## tuning for this so will change this
  "lr": 1e-3,
  "shuffle_obs_in_demo": False,
  "shuffle_data": False,
  # "lock_loader_seed": 42 ## TEST: test this later not sure
  # "lambda_grasp_loss": 20 ## 1 by default
}

task_params = {
  "scale" : 1.0, 
  "wrist_cam_distance": 0.5
}
repeats = 10

mb_sizes = [128, 256, 512]

In [ ]:
demos = task_env.get_demos(1, live_demos = True)

In [ ]:
combs = list(product(cam_types, mb_sizes, range(repeats)))
df = pd.DataFrame(columns=[
  "task_name", "cam_type", "demo_count", "max_eplen", 
  "mb_size",
  "is_success", "min_distance", "final_distance",
  "gripper_image_paths"
  
], index = range(len(combs))) 
df.to_csv("vision_single_static-hpt-results.csv", index=True)

In [ ]:
for i, (cam_type, mb, rep) in enumerate(combs):
  print(f"Running {get_task_name(task)} with {cam_type} and 1 demo, mb size: {mb} repeat: {rep}")

  agent = Agent(
    env.action_shape[0],
    PolicyType.SIMPLE_GRASP,
    cam_type,
    # grasp_thresh = 0.5, this is default
  ) 


  copy_params = training_params
  copy_params["minibatch_size"] = mb


  rets, done = run_grasp_with_agent(
    env, task, agent, demos, max_eplen = "demo_max", 
    training_params=copy_params, 
    task_params=task_params,
    print_index=i
  )
  print(f"{done = }")
  
  df.loc[i] = {
    "task_name": get_task_name(task),
    "cam_type": cam_type,
    "demo_count": 1,
    "max_eplen": "demo_max",
    "mb_size": mb,
    "is_success": done,
    "min_distance": min(rets["distances"]),
    "final_distance": rets["distances"][-1],
    "gripper_image_paths": rets["gripper_image_paths"]

  }
  df.to_csv("vision_single_static-hpt-results.csv", index=True)
                                        

## Param Tuning + Vision_Random

I also want to see generalisations, running wrist alone a few times, I saw that the wrist camera generalises better across multiple trials compared to a single wrist or shoulder camera

This will also be provide a better representation for which mb_size is actually better

In [ ]:
cam_types = [
  ## NOTE: not running the others yet, unless the wrist is actually better this time
  CamType.WRIST, 
  # CamType.RIGHT_SHOULDER,
  # CamType.LEFT_SHOULDER,
  # CamType.LEFT_SHOULDER | CamType.RIGHT_SHOULDER
]

task = Vision_Random
# size is still the same, but object will be randomly placed
task_env = env.get_task(task, scale = 1, wrist_cam_distance = 0.5)

training_params = {
  "epochs": 1000,
  "minibatch_size": None, ## tuning for this so will change this
  "lr": 1e-3,
  "shuffle_obs_in_demo": False,
  "shuffle_data": False,
  # "lock_loader_seed": 42 ## TEST: test this later not sure
  # "lambda_grasp_loss": 20 ## 1 by default
}

task_params = {
  "scale" : 1.0, 
  "wrist_cam_distance": 0.5 ## NOTE: should I vary the distance??
}

## want to run 5 trials per trained policy, and then train 10 policies per combination
## this will take time
train_repeats = 10
repeats = 5
demo_counts = [1, 10]
## demo lengths are around ~60, 
mb_sizes = [20, 30, 60] # [128, 256, 512]


In [ ]:
demos = task_env.get_demos(max(demo_counts), live_demos = True)

In [ ]:
demo_lens = [len(demo) for demo in demos]
mb_sizes = [sum(demo_lens) // len(demos), max(demo_lens)]
mb_sizes

In [ ]:
combs = list(
  product(
    cam_types,
    mb_sizes,
    demo_counts,
    range(repeats),
  )
)

df = pd.DataFrame(columns=[
  "task_name", 
  "setting_rep", ## repeat as before
  "train_rep", ## policy index (see `train_repeats`)
  "cam_type", 
  "demo_count", 
  "max_eplen", 
  "mb_size",
  "is_success",
  "min_distance",
  "final_distance",
  "gripper_image_paths"
  
], index = range(len(combs) * train_repeats)) 

df_path = "vision_single_random_fixed_size-hpt-results--dynamic-len.csv"

df.to_csv(df_path, index=True)

In [ ]:
for i, (cam_type, mb, demo_count, rep) in enumerate(combs):
  print(f"Running {get_task_name(task)} with {cam_type} and 1 demo, mb size: {mb} repeat: {rep}")

  agent = Agent(
    env.action_shape[0],
    PolicyType.SIMPLE_GRASP,
    cam_type,
    # grasp_thresh = 0.5, this is default
  ) 


  copy_params = training_params
  copy_params["minibatch_size"] = mb


  task_env, _ = demos_and_train_for_task(
    env, 
    task, 
    agent, 
    demos, 
    save_model= True,
    training_params=copy_params, 
    task_params=task_params
  )

  for train_rep in range(train_repeats):
    print(f"\t Trial {train_rep}:")
    rets, done = run_grasp_with_agent(
      env, task, agent, demos[:demo_count], max_eplen = "demo_max", 
      training_params=copy_params, 
      task_params=task_params,
      print_index=i,
      task_env=task_env, ## this forces NO TRAINING!!
    )
    print(f"{done = }")
    
    df.loc[(i * train_repeats) + train_rep] = {
      "task_name": get_task_name(task),
      "cam_type": cam_type,
      "setting_rep": i,
      "train_rep": train_rep,
      "demo_count": demo_count,
      "max_eplen": "demo_max",
      "mb_size": mb,
      "is_success": done,
      "min_distance": min(rets["distances"]),
      "final_distance": rets["distances"][-1],
      "gripper_image_paths": rets["gripper_image_paths"]

    }
    df.to_csv(df_path, index=True)
                                        

In [ ]:
## concat all these separate csvs

first_df = pd.read_csv("run-csvs/vision-exp/hpt-trials/vision_single_random_fixed_size-hpt-results-1.csv")
lr_df = pd.read_csv("run-csvs/vision-exp/hpt-trials/vision_single_random_fixed_size-hpt-results--l+r_s.csv")
l128_df = pd.read_csv("run-csvs/vision-exp/hpt-trials/vision_single_random_fixed_size-hpt-results--l_s-10-128.csv")
l_rest = pd.read_csv("run-csvs/vision-exp/hpt-trials/vision_single_random_fixed_size-hpt-results--l-rest.csv")#]


df = pd.concat([first_df, lr_df, l128_df, l_rest], axis = 0, ignore_index=True).reset_index()
# df.drop(["index", "Unamed: 0"], inplace=True)
df = df[["task_name", 
  "setting_rep", ## repeat as before
  "train_rep", ## policy index (see `train_repeats`)
  "cam_type", 
  "demo_count", 
  "max_eplen", 
  "mb_size",
  "is_success",
  "min_distance",
  "final_distance",
  "gripper_image_paths"]]
df.to_csv("vision_single_random_fixed_size-hpt-results.csv")
df

## Vision_Random - Trying "demo" dataset, entire demos, not mixed

See `plots.ipynb`, trying to run each demo once 

In [ ]:
cam_types = [
  ## NOTE: not running the others yet, unless the wrist is actually better this time
  CamType.WRIST, 
  # CamType.RIGHT_SHOULDER,
  # CamType.LEFT_SHOULDER,
  # CamType.LEFT_SHOULDER | CamType.RIGHT_SHOULDER
]

task = Vision_Random
# size is still the same, but object will be randomly placed
task_env = env.get_task(task, scale = 1, wrist_cam_distance = 0.5)

training_params = {
  "epochs": 1000,
  "minibatch_size": None, ## tuning for this so will change this
  "lr": 1e-3,
  "shuffle_obs_in_demo": False,
  "shuffle_data": True, ## TODO: run this agagin with shuffle
  "dataset_to_use": "demo" ## use demo less training
  # "lock_loader_seed": 42 ## TEST: test this later not sure
  # "lambda_grasp_loss": 20 ## 1 by default
}

task_params = {
  "scale" : 1.0, 
  "wrist_cam_distance": 0.5 ## NOTE: should I vary the distance??
}

## want to run 5 trials per trained policy, and then train 10 policies per combination
## this will take time
train_repeats = 10
repeats = 5
demo_counts = [10] ## not running 1 demo, doesn't mean anything, not gonna generalise anyway
## demo lengths are around ~60, 
epochs = [200]


In [ ]:
demos = task_env.get_demos(max(demo_counts), live_demos = True)

In [ ]:
combs = list(
  product(
    cam_types,
    epochs,
    demo_counts,
    range(repeats),
  )
)

df = pd.DataFrame(columns=[
  "task_name", 
  "setting_rep", ## repeat as before
  "train_rep", ## policy index (see `train_repeats`)
  "cam_type", 
  "demo_count", 
  "max_eplen", 
  "epochs",
  "is_success",
  "min_distance",
  "final_distance",
  "gripper_image_paths"
  
], index = range(len(combs) * train_repeats)) 

df_path = "vision_single_random_fixed_size-hpt-results--demo_dataset.csv"

df.to_csv(df_path, index=True)

In [ ]:
for i, (cam_type, epoch, demo_count, rep) in enumerate(combs):
  print(f"Running {get_task_name(task)} with {cam_type} and 1 demo, epoch: {epoch} repeat: {rep}")

  agent = Agent(
    env.action_shape[0],
    PolicyType.SIMPLE_GRASP,
    cam_type,
    # grasp_thresh = 0.5, this is default
  ) 


  copy_params = training_params
  copy_params["epochs"] = epoch

  ## trained here
  task_env, _ = demos_and_train_for_task(
    env, 
    task, 
    agent, 
    demos, 
    save_model= True,
    training_params=copy_params, 
    task_params=task_params
  )

  for train_rep in range(train_repeats):
    print(f"\t Trial {train_rep}:")
    rets, done = run_grasp_with_agent(
      env, task, agent, demos[:demo_count], max_eplen = "demo_max", 
      training_params=copy_params, 
      task_params=task_params,
      print_index=i,
      task_env=task_env, ## this forces NO TRAINING!!
    )
    
    df.loc[(i * train_repeats) + train_rep] = {
      "task_name": get_task_name(task),
      "cam_type": cam_type,
      "setting_rep": i,
      "train_rep": train_rep,
      "demo_count": demo_count,
      "max_eplen": "demo_max",
      "epochs": epoch,
      "is_success": done,
      "min_distance": min(rets["distances"]),
      "final_distance": rets["distances"][-1],
      "gripper_image_paths": rets["gripper_image_paths"]

    }
    df.to_csv(df_path, index=True)
                                        

#### entire demos but now they are being shuflled and multiple demos can be used to train
I want to keep the ratios similar I want around 2000 passes in the network
512 mb_size with 10 demos was about 2000  iterations, 

so now using lets say seq_size = 5 (which is 2 batches in 10 demos) want to do 1000 epochs, or seq = 2 (5) then 400 epochs etc, realsied with some mmanual trials that this network needs this scaling to work well

In [ ]:
cam_types = [
  CamType.WRIST, 
]

task = Vision_Random
# size is still the same, but object will be randomly placed
task_env = env.get_task(task, scale = 1, wrist_cam_distance = 0.5)

training_params = {
  "epochs": None,
  "minibatch_size": None, ## tuning for this so will change this
  "lr": 1e-3,
  "shuffle_obs_in_demo": False,
  "shuffle_data": True,
  "dataset_to_use": "demo" ## use demo less training
  # "lock_loader_seed": 42 ## TEST: test this later not sure
  # "lambda_grasp_loss": 20 ## 1 by default
}

task_params = {
  "scale" : 1.0, 
  "wrist_cam_distance": 0.5 ## NOTE: should I vary the distance??
}

## want to run 5 trials per trained policy, and then train 10 policies per combination
## this will take time
train_repeats = 10
repeats = 5
demo_counts = [10] ## not running 1 demo, doesn't mean anything, not gonna generalise anyway
## demo lengths are around ~60, 
pass_tot = 2000
mb_sizes = [2, 5, 10]


In [ ]:
demos = task_env.get_demos(max(demo_counts), live_demos = True)

In [ ]:
combs = list(
  product(
    cam_types,
    mb_sizes,
    demo_counts,
    range(repeats),
  )
)

df = pd.DataFrame(columns=[
  "task_name", 
  "setting_rep", ## repeat as before
  "train_rep", ## policy index (see `train_repeats`)
  "cam_type", 
  "demo_count", 
  "max_eplen", 
  "epochs", 
  "mb_size",
  "is_success",
  "min_distance",
  "final_distance",
  "gripper_image_paths"
  
], index = range(len(combs) * train_repeats)) 

df_path = "vision_single_random_fixed_size-hpt-results--demo_dataset-batch_collated.csv"

df.to_csv(df_path, index=True)

In [ ]:
for i, (cam_type, mb_size, demo_count, rep) in enumerate(combs):
  epoch = pass_tot // (10 // mb_size)
  print(f"Running {get_task_name(task)} with {cam_type} and 1 demo, mb_size: {mb_size}, epoch: {epoch} repeat: {rep}")

  agent = Agent(
    env.action_shape[0],
    PolicyType.SIMPLE_GRASP,
    cam_type,
    # grasp_thresh = 0.5, this is default
  ) 


  copy_params = training_params
  copy_params["epochs"] = epoch
  copy_params["minibatch_size"] = mb_size

  ## trained here
  task_env, _ = demos_and_train_for_task(
    env, 
    task, 
    agent, 
    demos, 
    save_model= True,
    training_params=copy_params, 
    task_params=task_params
  )

  for train_rep in range(train_repeats):
    print(f"\t Trial {train_rep}:")
    rets, done = run_grasp_with_agent(
      env, task, agent, demos[:demo_count], max_eplen = "demo_max", 
      training_params=copy_params, 
      task_params=task_params,
      print_index=i,
      task_env=task_env, ## this forces NO TRAINING!!
    )
    
    df.loc[(i * train_repeats) + train_rep] = {
      "task_name": get_task_name(task),
      "cam_type": cam_type,
      "setting_rep": i,
      "train_rep": train_rep,
      "demo_count": demo_count,
      "max_eplen": "demo_max",
      "epochs": epoch,
      "mb_size": mb_size, 
      "is_success": done,
      "min_distance": min(rets["distances"]),
      "final_distance": rets["distances"][-1],
      "gripper_image_paths": rets["gripper_image_paths"]

    }
    df.to_csv(df_path, index=True)
                                        

#### Quick test about using the other loader with same demos with the entirety of frames being selected for the batchsize



In [ ]:
cam_types = [
  CamType.WRIST, 
]

task = Vision_Random
task_params = {
  "scale" : 1.0, 
  "wrist_cam_distance": 0.5 ## NOTE: should I vary the distance??
}
# size is still the same, but object will be randomly placed
task_env = env.get_task(task, scale = 1, wrist_cam_distance = 0.5)

training_params = {
  "epochs": None,
  "minibatch_size": None, ## tuning for this so will change this
  "lr": 1e-3,
  "shuffle_obs_in_demo": False,
  "shuffle_data": False,
  "dataset_to_use": "obs" ## use demo less training
  # "lock_loader_seed": 42 ## TEST: test this later not sure
  # "lambda_grasp_loss": 20 ## 1 by default
}


## want to run 5 trials per trained policy, and then train 10 policies per combination
## this will take time
train_repeats = 10
repeats = 5
demo_counts = [10] ## not running 1 demo, doesn't mean anything, not gonna generalise anyway
## demo lengths are around ~60, 


In [ ]:
mb_sizes = [sum(map(len, demos))]
epochs = [2000]

In [ ]:
combs = list(
  product(
    cam_types,
    epochs,
    mb_sizes,
    demo_counts,
    range(repeats),
  )
)

df = pd.DataFrame(columns=[
  "task_name", 
  "setting_rep", ## repeat as before
  "train_rep", ## policy index (see `train_repeats`)
  "cam_type", 
  "demo_count", 
  "max_eplen", 
  "epochs", 
  "mb_size",
  "is_success",
  "min_distance",
  "final_distance",
  "gripper_image_paths"
  
], index = range(len(combs) * train_repeats)) 

df_path = "vision_single_random_fixed_size-hpt-results--demo_obs_dataset-entire_batch.csv"

df.to_csv(df_path, index=True)

In [ ]:
for i, (cam_type, epoch, mb_size, demo_count, rep) in enumerate(combs):
  print(f"Running {get_task_name(task)} with {cam_type} and 1 demo, mb_size: {mb_size}, epoch: {epoch} repeat: {rep}")

  agent = Agent(
    env.action_shape[0],
    PolicyType.SIMPLE_GRASP,
    cam_type,
    # grasp_thresh = 0.5, this is default
  ) 


  copy_params = training_params
  copy_params["epochs"] = epoch
  copy_params["minibatch_size"] = mb_size

  ## trained here
  task_env, _ = demos_and_train_for_task(
    env, 
    task, 
    agent, 
    demos, 
    save_model= True,
    training_params=copy_params, 
    task_params=task_params
  )

  for train_rep in range(train_repeats):
    print(f"\t Trial {train_rep}:")
    rets, done = run_grasp_with_agent(
      env, task, agent, demos[:demo_count], max_eplen = "demo_max", 
      training_params=copy_params, 
      task_params=task_params,
      print_index=i,
      task_env=task_env, ## this forces NO TRAINING!!
    )
    
    df.loc[(i * train_repeats) + train_rep] = {
      "task_name": get_task_name(task),
      "cam_type": cam_type,
      "setting_rep": i,
      "train_rep": train_rep,
      "demo_count": demo_count,
      "max_eplen": "demo_max",
      "epochs": epoch,
      "mb_size": mb_size, 
      "is_success": done,
      "min_distance": min(rets["distances"]),
      "final_distance": rets["distances"][-1],
      "gripper_image_paths": rets["gripper_image_paths"]

    }
    df.to_csv(df_path, index=True)
                                        

# Depth Interfacing (adding the Wrist Depth Camera)

3 combinations to try:
1. add depth as an extra channel let the same network take care of the understanding
1. add a secondary depth cnn, which can later be infused with the featues from the rgb cnns (still `channel * len(cam_type)`) either with:
    - incorporate into later heads, let `action_head` and `grasp_head` take care of by expanding their inputs 
    - separate fuser linear network, extra `self.feat_fuser()`
1. Make each camera have its own encoder, like `CamAttentionPolicy` (was having different feats good here I forgot?) and then a mechanism to merge and use them later on
with the different heads (each head still makes its decision form that feature combination, not sure how to combine the features yet)

Created a new policy class `DepthGraspPolicy` to test. The above can be configured by passing a different config to the agent in creation which will propagate down and initialise the network accordingly

## 1 - Depth as Extra Channel
added back the loader_seed variable so I can control exacly how the data is shuffled

Experimment set up: I want to see if the wrist cam or wrist cam + wrist depth performs better in random placed task?

First I want to try training them on the same dataset (controlling the shuffling)
then reset the environment the demos and see their performance on the same demos and placed objects. 

Test 2 will be controlling the test demos with task sizes etc


The network is bad at learning from a single demonstration. Seems to not be deep enough so I want to train on 10 demos randomising them as before then rolling out a system to test them on different size objects


In [ ]:
task = Vision_Random

## Train on 1 to 1 scale and fixed distance
train_task_params = {
  "scale" : 1.0, 
  "wrist_cam_distance": 0.6 
}
task_env = env.get_task(task, **train_task_params)
train_demos = task_env.get_demos(10, live_demos=True)


## test on half the distance but smaller
test_task_params = {
  "scale" : 0.5, 
  "wrist_cam_distance": 0.3 
}
task_env = env.get_task(task, **test_task_params)
test_demos = task_env.get_demos(10, live_demos=True)

##TODO do the opposite later
## TODO: vary the distance the boxes are generated

In [ ]:
training_params = {
  "epochs": 2000,
  "minibatch_size": 10,
  "shuffle_data": True,
  "lock_loader_seed": 12, ##lock the randomisation of training samples
  "lr": 1e-3,
}

## setup our 2 agents
agent_rgb = lambda: Agent(
  env.action_shape[0],
  policy_type=PolicyType.DEPTH_GRASP,
  cam_type=CamType.WRIST, 
  config = "depth_ch"
)
agent_depth = lambda: Agent(
  env.action_shape[0],
  policy_type=PolicyType.DEPTH_GRASP,
  cam_type=CamType.WRIST | CamType.WRIST_DEPTH, 
  config = "depth_ch"
)

In [ ]:
frame = train_and_test_vision_config(
  env, 
  [agent_rgb(), agent_depth()],
  [training_params, training_params], 
  train_demos,
  test_demos,
  task, 
  train_task_params, 
  test_task_params, 
  df_csv_name=f"depth_policy-wrist-vs-wrist+wrist_depth-comparison-2000ep-10mbsize-12loaderseed-001lr",
  check_random_demos=1,
  save_individual= False
)


In [ ]:
for seed in np.random.randint(0, high = 10000, size = (10)):
  training_params["lock_loader_seed"] = seed
  
  frame = train_and_test_vision_config(
    env, 
    [agent_rgb(), agent_depth()],
    [training_params, training_params], 
    train_demos,
    test_demos,
    task, 
    train_task_params, 
    test_task_params, 
    df_csv_name=f"depth_policy-wrist-vs-wrist+wrist_depth-comparison-2000ep-10mbsize-12loaderseed-001lr",
    check_random_demos=1,
    save_individual= False
  )




In [ ]:
# task_env.reset(demo = demos[0] )
# task_env.reset()
task_env.reset_to_demo(demos[0])


## 2- Depth with its own Conv
Now there are two options to fuse:
1. The itemms are fused using a separate MLP and the output of this MLP us used to
1. the fuser learns what to keep and what to discard, acts as a gate



In [ ]:
task = Vision_Random

## Train on 1 to 1 scale and fixed distance
train_task_params = {
  "scale" : 0.5, 
  "wrist_cam_distance": 0.3 
}
task_env = env.get_task(task, **train_task_params)
train_demos = task_env.get_demos(10, live_demos=True)


## test on half the distance but smaller
test_task_params = {
  "scale" : 1, 
  "wrist_cam_distance": 0.6
}
task_env = env.get_task(task, **test_task_params)
test_demos = task_env.get_demos(10, live_demos=True)

save_demos(train_demos, "train_demos")
save_demos(test_demos, "test_demos")
##TODO do the opposite later
## TODO: vary the distance the boxes are generated

In [ ]:
training_params2000ep = {
  "epochs": 2000,
  "minibatch_size": 10,
  "shuffle_data": True,
  "lock_loader_seed": None, ## the loop will choose this
  "lr": 1e-3,
}
# training_params1500ep = {
#   "epochs": 1500,
#   "minibatch_size": 10,
#   "shuffle_data": True,
#   "lock_loader_seed": None, ## the loop will choose this
#   "lr": 1e-3,
# }

## setup our agents: Control and cated depth channel
agent_rgb = lambda: Agent(
  env.action_shape[0],
  policy_type=PolicyType.DEPTH_GRASP,
  cam_type=CamType.WRIST, 
  config = "depth_ch"
)
agent_rgb_l_r_ch = lambda: Agent(
  env.action_shape[0],
  policy_type=PolicyType.DEPTH_GRASP,
  cam_type=CamType.LEFT_SHOULDER | CamType.RIGHT_SHOULDER, 
  config = "depth_ch"
)
agent_depth_ch = lambda: Agent(
  env.action_shape[0],
  policy_type=PolicyType.DEPTH_GRASP,
  cam_type=CamType.WRIST | CamType.WRIST_DEPTH, 
  config = "depth_ch"
)

### Learnt Fuser with `depth_feats`


In [ ]:
agent_depth_feats = lambda: Agent(
  env.action_shape[0],
  policy_type=PolicyType.DEPTH_GRASP,
  cam_type=CamType.WRIST | CamType.WRIST_DEPTH, 
  config = "depth_feats",
  opts = {"gated_fuse": False}
)

agent_depth_feats_l_r = lambda: Agent(
  env.action_shape[0],
  policy_type=PolicyType.DEPTH_GRASP,
  cam_type=CamType.LEFT_SHOULDER | CamType.RIGHT_SHOULDER | CamType.WRIST_DEPTH, 
  config = "depth_feats",
  opts = {"gated_fuse": False}
)


### Gated Feats with `depth_feats` 
uses `opts["gated_fuse"] = True` option to create the policy

In [ ]:
agent_depth_feats_gate = lambda: Agent(
  env.action_shape[0],
  policy_type=PolicyType.DEPTH_GRASP,
  cam_type=CamType.WRIST | CamType.WRIST_DEPTH, 
  config = "depth_feats",
  opts = {"gated_fuse": True}
)

agent_depth_feats_gate_l_r = lambda: Agent(
  env.action_shape[0],
  policy_type=PolicyType.DEPTH_GRASP,
  cam_type=CamType.LEFT_SHOULDER | CamType.RIGHT_SHOULDER | CamType.WRIST_DEPTH, 
  config = "depth_feats",
  opts = {"gated_fuse": True}
)

In [ ]:
seeds = np.random.randint(0, high = 10000, size = (5))


for seed in seeds:
  
  name = f"depth_feats--4way-cam-and-gate-comparison-2000ep-10mbsize-{seed}loaderseed-001lr"
  training_params2000ep["lock_loader_seed"] = seed

  agents = [
    agent_depth_feats(),
    agent_depth_feats_l_r(),
    agent_depth_feats_gate(),
    agent_depth_feats_gate_l_r(),
  ]

  ag_params = [training_params2000ep] * len(agents)

  frame, rets = train_and_test_vision_config(
    env, 
    agents,
    ag_params, 
    test_demos,
    train_demos,
    task, 
    test_task_params, 
    train_task_params, 
    df_csv_name=name,
    check_random_demos=10,
    save_individual= True
  )
  try:
    save_demos(rets["random_demos"], f"{seed}-random_demos")
  except:
    print("Error saving demos, dont really care")
seeds

In [ ]:
seeds = np.random.randint(0, high = 10000, size = (10))
for seed in seeds:
  
  name = f"rerun--small_then_normal-depth_policy-5way-comparison-2000ep-10mbsize-{seed}loaderseed-001lr"
  training_params2000ep["lock_loader_seed"] = seed

  agents = [
    agent_rgb(),
    agent_rgb_l_r_ch(),
    agent_depth_ch(),
    agent_depth_feats(),
    agent_depth_feats_gate()
  ]

  ag_params = [training_params2000ep] * len(agents)

  frame, rets = train_and_test_vision_config(
    env, 
    agents,
    ag_params, 
    train_demos,
    test_demos,
    task, 
    train_task_params, 
    test_task_params, 
    df_csv_name=name,
    check_random_demos=10,
    save_individual= True
  )
  try:
    save_demos(rets["random_demos"], f"{seed}-random_demos--small_then_normal")
  except:
    print("Error saving demos, dont really care")
  # break
seeds

In [ ]:
agent = agent_depth_feats_gate()
# task_env = env.get_task(task, **train_task_params)
_, obs = task_env.reset()
count = 0
done = False
distances = []
while not done:
  
  action, _ = agent.act(obs)
  print(f"{action = }")

  action = action.squeeze(0)
  print(f"{action.shape =}")
  print(f"{action[-1] =}")
  # print(f"{action.shape = }")
  obs, reward, done = task_env.step(action)
  gripper = Object.get_object("Panda_gripper")
  target = Object.get_object("grasp_cube")

  
  
  # vis_score = check_visibility("cam_wrist", "target")
  
  # print(f"{vis_score = }")
  
  
  distance = np.linalg.norm(gripper.get_position() - target.get_position())
  distances.append(distance)
  # print(f"{done = }")
  
  count += 1
  if count == 100:
    break
  
print(f"Done Successfull! done in {count} steps" if done else "Failed!")
print(f"Final distance: {distances[-1]}")


## Trying a Resnet backbone for instead of a simple Conv
This is mainly a bridge before I get to LSTM, because ResNet doesn't seem to help though I can see that it might be useful to have residual connections about previous states of the features/cameras

In [ ]:
task = Vision_Random

## Train on 1 to 1 scale and fixed distance
train_task_params = {
  "scale" : 1, 
  "wrist_cam_distance": 0.6
}
task_env = env.get_task(task, **train_task_params)
train_demos = task_env.get_demos(10, live_demos=True)
save_demos(train_demos, "train_demos")

test_demos = task_env.get_demos(10, live_demos=True)
save_demos(test_demos, "test_demos")

In [ ]:
train_params_cnn = {
  "epochs": 2000,
  "minibatch_size": 10,
  "shuffle_data": True,
  "lock_loader_seed": None, ## the loop will choose this
  "lr": 1e-3,
}
train_params_resnet = {
  "epochs": 400, ## the resnet variants learn better with less epochs
  "minibatch_size": 10,
  "shuffle_data": True,
  "lock_loader_seed": None, ## the loop will choose this
  "lr": 1e-3,
}
 

### Checking Wrist and Wrist Depth

In [ ]:
resnets = ["resnet18", "resnet34", "resnet50"]
seeds = np.random.randint(0, high = 10000, size = (3))

cam_type = CamType.WRIST | CamType.WRIST_DEPTH

for seed in seeds:
  name = f"4way-resnetcomp-cam:{cam_type}-400or2000ep-10mbsize-{seed}loaderseed-001lr"

  agents = [Agent(
    env.action_shape[0],
    policy_type=PolicyType.RESNET_GRASP,
    cam_type=cam_type,
    config="depth_feats",
    opts= {
      "gated_fuse": True,
      "resnet_name": resnet_name,
      "kernel_size": 3 
    },
  ) for resnet_name in resnets]

  params = [train_params_resnet] * len(agents)

  conv_agent = Agent(
    env.action_shape[0],
    policy_type=PolicyType.DEPTH_GRASP,
    cam_type=cam_type,
    config = "depth_feats",
    opts = {"gated_fuse": True}
  )

  agents.append(conv_agent)
  params.append(train_params_cnn)

  for param in params:
    param["lock_loader_seed"] = seed

  frame, rets = train_and_test_vision_config(
    env, 
    agents,
    params, 
    train_demos,
    test_demos,
    task, 
    train_task_params, 
    train_task_params, 
    df_csv_name=name,
    check_random_demos=0,
    save_individual= True
)


### Checking for L+R Cameras

In [ ]:
resnets = ["resnet18", "resnet34", "resnet50"]
seeds = np.random.randint(0, high = 10000, size = (3))

cam_type = CamType.LEFT_SHOULDER | CamType.RIGHT_SHOULDER

for seed in seeds:
  name = f"4way-resnetcomp-cam:{cam_type}-400or2000ep-10mbsize-{seed}loaderseed-001lr"

  agents = [Agent(
    env.action_shape[0],
    policy_type=PolicyType.RESNET_GRASP,
    cam_type=cam_type,
    config="depth_ch",
    opts= {
      "gated_fuse": True,
      "resnet_name": resnet_name,
      "kernel_size": 3 
    },
  ) for resnet_name in resnets]

  params = [train_params_resnet] * len(agents)

  conv_agent = Agent(
    env.action_shape[0],
    policy_type=PolicyType.DEPTH_GRASP,
    cam_type=cam_type,
    config = "depth_ch",
    opts = {"gated_fuse": True}
  )

  agents.append(conv_agent)
  params.append(train_params_cnn)
  
  for param in params:
    param["lock_loader_seed"] = seed

  frame, rets = train_and_test_vision_config(
    env, 
    agents,
    params, 
    train_demos,
    test_demos,
    task, 
    train_task_params, 
    train_task_params, 
    df_csv_name=name,
    check_random_demos=0,
    save_individual= True
)


# Sequential understanding of the Demos
Transformer or LSTM (or equivalent) to index the sequences so they can be differentiated, in terms of indexing in the demo, this means that the system can learn the difference between where it is in the sequence, might help with the grasp? 
-> if transformer, understand how the manual indexing is done (unlike an LSTM) the data is not inherently sequential so there needs to be manual indexing


## LSTM - RNNs


In [ ]:
task = Vision_Random

## Train on 1 to 1 scale and fixed distance
normal_size_task_params = {
  "scale" : 1.0, 
  "wrist_cam_distance": 0.6
}
task_env = env.get_task(task, **normal_size_task_params)
normal_demos = task_env.get_demos(10, live_demos=True)


## test on half the distance but smaller
smaller_task_params = {
  "scale" : 0.5, 
  "wrist_cam_distance": 0.3
}
task_env = env.get_task(task, **smaller_task_params)
smaller_demos = task_env.get_demos(10, live_demos=True)

save_demos(normal_demos, "normal_demos")
save_demos(smaller_demos, "smaller_demos")

In [ ]:
# training_params200ep = {
#   "epochs": 200,
#   "minibatch_size": 10,
#   "shuffle_data": True,
#   "lock_loader_seed": None, ## the loop will choose this
#   "lr": 1e-3,
# }
training_params400ep = {
  "epochs": 400,
  "minibatch_size": 10,
  "shuffle_data": True,
  "lock_loader_seed": None, ## the loop will choose this
  "lr": 1e-3,
}

training_params600ep = {
  "epochs": 600,
  "minibatch_size": 10,
  "shuffle_data": True,
  "lock_loader_seed": None, ## the loop will choose this
  "lr": 1e-3,
}
# training_params1000ep = {
#   "epochs": 1000,
#   "minibatch_size": 10,
#   "shuffle_data": True,
#   "lock_loader_seed": None, ## the loop will choose this
#   "lr": 1e-3,
# }

# training_params2000ep = {
#   "epochs": 2000,
#   "minibatch_size": 10,
#   "shuffle_data": True,
#   "lock_loader_seed": None, ## the loop will choose this
#   "lr": 1e-3,
# }

## setup our agents: Control and cated depth channel


In [ ]:
## gated fuse - depth_ch
agent_rnn_wrist_rgb= lambda: Agent(
  env.action_shape[0],
  policy_type=PolicyType.RNN_GRASP,
  cam_type=CamType.WRIST, 
)

agent_rnn_wdepth = lambda: Agent(
  env.action_shape[0],
  policy_type=PolicyType.RNN_GRASP,
  cam_type=CamType.WRIST | CamType.WRIST_DEPTH, 
)

agent_rnn_l_r = lambda: Agent(
  env.action_shape[0],
  policy_type=PolicyType.RNN_GRASP,
  cam_type= CamType.LEFT_SHOULDER | CamType.RIGHT_SHOULDER ,
)

agent_rnn_l_r_wd = lambda: Agent(
  env.action_shape[0],
  policy_type=PolicyType.RNN_GRASP,
  cam_type= CamType.LEFT_SHOULDER | CamType.RIGHT_SHOULDER  | CamType.WRIST_DEPTH,
)

agent_rnn_r_wd = lambda: Agent(
  env.action_shape[0],
  policy_type=PolicyType.RNN_GRASP,
  cam_type= CamType.RIGHT_SHOULDER  | CamType.WRIST_DEPTH,
)

In [ ]:
seeds = np.random.randint(0, high = 10000, size = (5))
# ## uusing the same seeds as the first half, crashed run

# # seeds = [
# #   1255, 3370, 3486, 3944, 8614, 
# # ]

# for param in [ training_params1000ep, training_params2000ep ]:

## this is found to be the one that works the best?
param = training_params600ep

for seed in seeds:
  for param in [training_params400ep, training_params600ep]:
    name = f"400600--5way-rnncomp-2wdepth_2wo-{param['epochs']}ep-10mbsize-{seed}loaderseed-001lr"

    param["lock_loader_seed"] = seed

    agents = [    
      agent_rnn_wrist_rgb(),
      agent_rnn_wdepth(),
      agent_rnn_l_r(),
      agent_rnn_r_wd(),
      agent_rnn_l_r_wd()
    ]

    params = [param] * len(agents)

    frame, rets = train_and_test_vision_config(
      env, 
      agents,
      params, 
      normal_demos,
      smaller_demos,
      task, 
      normal_size_task_params, 
      smaller_task_params, 
      df_csv_name=name,
      check_random_demos=10,
      save_individual= True
      
    )
    try:
      save_demos(rets["random_demos"], f"{seed}-random_demos")
    except:
      print("Error saving demos, dont really care")


In [ ]:
sorted(seeds)
seeds

Train on small then on small targets then execute on the normal ones

In [ ]:
# seeds = np.random.randint(0, high = 10000, size = (5))
## uusing the same seeds as the first half, crashed run

seeds = [
  1255, 3370, 3486, 3944, 8614, 
]
## will train on the smaller boxes then test on the normal size
for param in [ training_params200ep, training_params400ep, training_params600ep ]:
  for seed in seeds:
    name = f"small_then_normal--4way-rnncomp-{param['epochs']}ep-10mbsize-{seed}loaderseed-001lr"

    param["lock_loader_seed"] = seed

    agents = [
      agent_rnn_wrist_rgb(),
      agent_rnn_wdepth(),
      agent_rnn_l_r(),
      agent_rnn_l_r_wd()
    ]

    params = [param] * len(agents)

    frame, rets = train_and_test_vision_config(
      env, 
      agents,
      params, 
      smaller_demos,
      normal_demos,
      task, 
      smaller_task_params, 
      normal_size_task_params, 
      df_csv_name=name,
      check_random_demos=10,
      save_individual= True
  )


In [ ]:
seeds

## Adding attention to LSTMs
compare if adding attention to the network makes a meaningful difference

In [ ]:
task = Vision_Random

## Train on 1 to 1 scale and fixed distance
normal_size_task_params = {
  "scale" : 1.0, 
  "wrist_cam_distance": 0.6
}
task_env = env.get_task(task, **normal_size_task_params)
normal_demos = task_env.get_demos(10, live_demos=True)


## test on half the distance but smaller
smaller_task_params = {
  "scale" : 0.5, 
  "wrist_cam_distance": 0.3
}
task_env = env.get_task(task, **smaller_task_params)
smaller_demos = task_env.get_demos(10, live_demos=True)

save_demos(normal_demos, "normal_demos")
save_demos(smaller_demos, "smaller_demos")

In [ ]:
training_params400ep = {
  "epochs": 400,
  "minibatch_size": 10,
  "shuffle_data": True,
  "lock_loader_seed": None, ## the loop will choose this
  "lr": 1e-3,
}

training_params600ep = {
  "epochs": 600,
  "minibatch_size": 10,
  "shuffle_data": True,
  "lock_loader_seed": None, ## the loop will choose this
  "lr": 1e-3,
}

non- attn agents

In [ ]:
agent_rnn_wrist_rgb= lambda: Agent(
  env.action_shape[0],
  policy_type=PolicyType.RNN_GRASP,
  cam_type=CamType.WRIST, 
  # rnn_opts = None ## cam pass a dist {} to configure the underlying rnn
)

agent_rnn_wdepth = lambda: Agent(
  env.action_shape[0],
  policy_type=PolicyType.RNN_GRASP,
  cam_type=CamType.WRIST | CamType.WRIST_DEPTH, 
)

agent_rnn_l_r = lambda: Agent(
  env.action_shape[0],
  policy_type=PolicyType.RNN_GRASP,
  cam_type= CamType.LEFT_SHOULDER | CamType.RIGHT_SHOULDER ,
)

agent_rnn_l_r_wd = lambda: Agent(
  8,
  policy_type=PolicyType.RNN_GRASP,
  cam_type= CamType.LEFT_SHOULDER | CamType.RIGHT_SHOULDER  | CamType.WRIST_DEPTH,
)

agent_rnn_r_wd = lambda: Agent(
  env.action_shape[0],
  policy_type=PolicyType.RNN_GRASP,
  cam_type= CamType.RIGHT_SHOULDER  | CamType.WRIST_DEPTH,
)

attn agents

In [ ]:
## cant have attention without depth camera
agent_rnn_wdepth_attn = lambda: Agent(
  env.action_shape[0],
  policy_type=PolicyType.RNN_GRASP,
  cam_type=CamType.WRIST | CamType.WRIST_DEPTH, 
  rnn_opts = {
    "config": "attn"
  }
)

agent_rnn_l_r_wd_attn = lambda: Agent(
  env.action_shape[0],
  policy_type=PolicyType.RNN_GRASP,
  cam_type= CamType.LEFT_SHOULDER | CamType.RIGHT_SHOULDER  | CamType.WRIST_DEPTH,
  rnn_opts = {
    "config": "attn"
  }
)

agent_rnn_r_wd_attn = lambda: Agent(
  env.action_shape[0],
  policy_type=PolicyType.RNN_GRASP,
  cam_type= CamType.RIGHT_SHOULDER  | CamType.WRIST_DEPTH,
  rnn_opts = {
    "config": "attn"
  }
)

In [ ]:
seeds = np.random.randint(0, high = 10000, size = (5))

param = training_params400ep


for seed in seeds:
  name = f"6way-rnn_attn_comp-{param['epochs']}ep-10mbsize-{seed}loaderseed-001lr"

  param["lock_loader_seed"] = seed

  agents = [
    agent_rnn_wdepth(),
    agent_rnn_wdepth_attn(),
    agent_rnn_r_wd(),
    agent_rnn_r_wd_attn(),
    agent_rnn_l_r_wd(),
    agent_rnn_l_r_wd_attn()
  ]

  params = [param] * len(agents)

  frame, rets = train_and_test_vision_config(
    env, 
    agents,
    params, 
    normal_demos,
    smaller_demos,
    task, 
    normal_size_task_params, 
    smaller_task_params, 
    df_csv_name=name,
    check_random_demos=10,
    save_individual= True
    
  )
  try:
    save_demos(rets["random_demos"], f"{seed}-random_demos")
  except:
    print("Error saving demos, dont really care")


# Attention Backbone with CNN (`this is above sequential understanding in the report timeline`)
Here I tried to use multihead attention and wider feature representations to  meaningfully merge the rgb and depth information to extract more information form both and better meaningfully connect them to maybe utilise this informmation better

Cross attentio between depth and rgb features for deeper meaningful information extraction

Two systems: deep fuse and normal:
1. Normal learns how the rgb attends with respect to depth, and fuses these 
1. Deep fuse, after fusing more colvolutions are done like the original cnn backbone, 

In [ ]:
task = Vision_Random

## Train on 1 to 1 scale and fixed distance
normal_size_task_params = {
  "scale" : 1.0, 
  "wrist_cam_distance": 0.6
}
task_env = env.get_task(task, **normal_size_task_params)
normal_demos = task_env.get_demos(10, live_demos=True)


## test on half the distance but smaller
smaller_task_params = {
  "scale" : 0.5, 
  "wrist_cam_distance": 0.3
}
task_env = env.get_task(task, **smaller_task_params)
smaller_demos = task_env.get_demos(10, live_demos=True)

save_demos(normal_demos, "normal_demos")
save_demos(smaller_demos, "smaller_demos")

In [ ]:
training_params500ep = {
  "epochs": 500,
  "minibatch_size": 10,
  "shuffle_data": True,
  "lock_loader_seed": None, ## the loop will choose this
  "lr": 1e-3,
}
training_params1000ep = {
  "epochs": 1000,
  "minibatch_size": 10,
  "shuffle_data": True,
  "lock_loader_seed": None, ## the loop will choose this
  "lr": 1e-3,
}
training_params2000ep = {
  "epochs": 2000,
  "minibatch_size": 10,
  "shuffle_data": True,
  "lock_loader_seed": None, ## the loop will choose this
  "lr": 1e-3,
}

## setup our agents: Control and cated depth channel


In [ ]:
agent_attn_w_wd = lambda: Agent(
  env.action_shape[0],
  policy_type=PolicyType.DEPTH_GRASP,
  cam_type=CamType.WRIST | CamType.WRIST_DEPTH, 
  grasp_thresh = 0.5,
  config = "attn",
  opts = {
  }
)

agent_attn_deep_w_wd = lambda: Agent(
  env.action_shape[0],
  policy_type=PolicyType.DEPTH_GRASP,
  cam_type=CamType.WRIST | CamType.WRIST_DEPTH, 
  grasp_thresh = 0.5,
  config = "attn",
    opts = {
    "attn_deep_fuse": True,
  }
)
agent_attn_deep_l_r_wd = lambda: Agent(
  env.action_shape[0],
  policy_type=PolicyType.DEPTH_GRASP,
  cam_type=CamType.LEFT_SHOULDER | CamType.RIGHT_SHOULDER | CamType.WRIST_DEPTH, 
  grasp_thresh = 0.5,
  config = "attn",
    opts = {
    "attn_deep_fuse": True,
  }
)


In [ ]:
seeds = np.random.randint(0, high = 10000, size = (3))
for params in [training_params1000ep]:
  for seed in seeds:
    
    name = f"attn-3way-comparison-{params['epochs']}ep-10mbsize-{seed}loaderseed-001lr"
    params["lock_loader_seed"] = seed

    agents = [
      agent_attn_w_wd(),
      agent_attn_deep_w_wd(),
      agent_attn_deep_l_r_wd()
    ]

    ag_params = [params] * len(agents)

    frame, rets = train_and_test_vision_config(
      env, 
      agents,
      ag_params, 
      normal_demos,
      smaller_demos,
      task, 
      normal_size_task_params, 
      smaller_task_params, 
      df_csv_name=name,
      check_random_demos=10,
      save_individual= True
    )
    save_demos(rets["random_demos"], f"{seed}-random_demos")

  seeds

### Possible Issues:

- I haven't spent enough time tuning for these specific models, just some general sweeps of epochs batch sizes and lrs manually, no grid seraching or systematic testing (would have taken too long, for something that wasn't promising at all)

- Distribution and misalignment
    - Nodality mismatch cross attention fails to optimally 'align' the data between depth and rgb? not necessarily correlating data

    - Cross attention may not be respecting the geometric relationship between the features (different rgb views possible)
- Simplicity of gating might be superior here when the depth information or rgb info is irrelevant.
    - from watching the network perform we can cleraly see that (as with the earlier reaching task tests, colour is important in learning how to move to a specific patch in the workspace) however depth can be more useful near the end of the episode, with the network unaware of any timem sequencing (As the LSTM based network was not prommising and not working well) 
    - Time sensitive gating might have been promising, not sure how but with the failure of the LSTM based network i didn't pay too much attention to this.

# Big Depth Interfacing Run with everything so far

Trying many cam combinations and differnt agents with different settings for a big comparison

In [ ]:
## ======= depth_ch combinations
agent_rgb_ch = Agent(
  env.action_shape[0],
  policy_type=PolicyType.DEPTH_GRASP,
  cam_type=CamType.WRIST, 
  config = "depth_ch"
)
agent_rgb_l_r_ch = Agent(
  env.action_shape[0],
  policy_type=PolicyType.DEPTH_GRASP,
  cam_type=CamType.LEFT_SHOULDER | CamType.RIGHT_SHOULDER, 
  config = "depth_ch"
)
agent_depth_ch = Agent(
  env.action_shape[0],
  policy_type=PolicyType.DEPTH_GRASP,
  cam_type=CamType.WRIST | CamType.WRIST_DEPTH, 
  config = "depth_ch"
)

## shoulder cams and wrist depth
agent_rgb_l_r_wd_ch = Agent(
  env.action_shape[0],
  policy_type=PolicyType.DEPTH_GRASP,
  cam_type=CamType.LEFT_SHOULDER | CamType.RIGHT_SHOULDER | CamType.WRIST_DEPTH, 
  config = "depth_ch"
)


## gated fuse - depth_feats - wrist rgb and depth
agent_depth_feats_gate = Agent(
  env.action_shape[0],
  policy_type=PolicyType.DEPTH_GRASP,
  cam_type=CamType.WRIST | CamType.WRIST_DEPTH, 
  config = "depth_feats",
  opts = {"gated_fuse": True}
)

## gated fuse - depth_feats - l+r shoulder rgb and wrist depth
agent_depth_feats_gate_l_r = Agent(
  env.action_shape[0],
  policy_type=PolicyType.DEPTH_GRASP,
  cam_type= CamType.LEFT_SHOULDER | CamType.RIGHT_SHOULDER | CamType.WRIST_DEPTH, 
  config = "depth_feats",
  opts = {"gated_fuse": True}
)
## NOTE: discarding ResNet policy, verey lackluster will jsut waste my time running it
## Adding one LSTM one, it is not the best so dont care too much will do 2 variants
agent_depth_feats_gate_l_r = Agent(
  env.action_shape[0],
  policy_type=PolicyType.DEPTH_GRASP,
  cam_type= CamType.LEFT_SHOULDER | CamType.RIGHT_SHOULDER | CamType.WRIST_DEPTH, 
  config = "depth_feats",
  opts = {"gated_fuse": True}
)


# some are changed and more to be added I think

# Including Proprioceptive information
Trying some policies with and without the proprioception stuff

In [ ]:
task = Vision_Random

## Train on 1 to 1 scale and fixed distance
normal_size_task_params = {
  "scale" : 1.0, 
  "wrist_cam_distance": 0.6
}
task_env = env.get_task(task, **normal_size_task_params)
normal_demos = task_env.get_demos(10, live_demos=True)


## test on half the distance but smaller
smaller_task_params = {
  "scale" : 0.5, 
  "wrist_cam_distance": 0.3
}
task_env = env.get_task(task, **smaller_task_params)
smaller_demos = task_env.get_demos(10, live_demos=True)

save_demos(normal_demos, "normal_demos")
save_demos(smaller_demos, "smaller_demos")

In [ ]:
training_params400ep = {
  "epochs": 400,
  "minibatch_size": 10,
  "shuffle_data": True,
  "lock_loader_seed": None, ## the loop will choose this
  "lr": 1e-3,
}

In [ ]:
agent_rnn_wrist_rgb = lambda: Agent(
  env.action_shape[0],
  policy_type=PolicyType.RNN_GRASP,
  cam_type=CamType.WRIST, 
  rnn_opts = {
    "config": "depth_feats",
    "use_proprio": False
  }
)

agent_rnn_wrist_rgb_proprio = lambda: Agent(
  env.action_shape[0],
  policy_type=PolicyType.RNN_GRASP,
  cam_type=CamType.WRIST, 
  rnn_opts = {
    "config": "depth_feats",
    "use_proprio": True
  }
)

agent_rnn_wd_feats= lambda: Agent(
  env.action_shape[0],
  policy_type=PolicyType.RNN_GRASP,
  cam_type=CamType.WRIST | CamType.WRIST_DEPTH, 
  rnn_opts = {
    "config": "depth_feats",
    "use_proprio": False
  }
)

agent_rnn_wd_feats_proprio = lambda: Agent(
  env.action_shape[0],
  policy_type=PolicyType.RNN_GRASP,
  cam_type=CamType.WRIST | CamType.WRIST_DEPTH, 
  rnn_opts = {
    "config": "depth_feats",
    "use_proprio": True
  }
)

agent_rnn_wrist_rgb_attn = lambda: Agent(
  env.action_shape[0],
  policy_type=PolicyType.RNN_GRASP,
  cam_type=CamType.WRIST | CamType.WRIST_DEPTH, 
  rnn_opts = {
    "config": "attn",
    "attn_opts": {"is_deep_fuse": True},
    "use_proprio": False
  }
)

agent_rnn_wrist_rgb_attn_proprio = lambda: Agent(
  env.action_shape[0],
  policy_type=PolicyType.RNN_GRASP,
  cam_type=CamType.WRIST | CamType.WRIST_DEPTH, 
  rnn_opts = {
    "config": "attn",
    "attn_opts": {"is_deep_fuse": True},
    "use_proprio": True
  }
)


In [ ]:
seeds = np.random.randint(0, high = 10000, size = (5))
# ## uusing the same seeds as the first half, crashed run

# # seeds = [
# #   1255, 3370, 3486, 3944, 8614, 
# # ]

# for param in [ training_params1000ep, training_params2000ep ]:

## this is found to be the one that works the best?
param = training_params400ep

for seed in seeds:
    name = f"proprio-check--2*2way-rnncomp{param['epochs']}ep-10mbsize-{seed}loaderseed-001lr"

    param["lock_loader_seed"] = seed

    agents = [    
      agent_rnn_wrist_rgb(),
      agent_rnn_wrist_rgb_proprio(),
      agent_rnn_wd_feats(),
      agent_rnn_wd_feats_proprio(),
      # agent_rnn_wrist_rgb_attn(),
      # agent_rnn_wrist_rgb_attn_proprio(),
    ]

    params = [param] * len(agents)

    frame, rets = train_and_test_vision_config(
      env, 
      agents,
      params, 
      normal_demos,
      smaller_demos,
      task, 
      normal_size_task_params, 
      smaller_task_params, 
      df_csv_name=name,
      check_random_demos=10,
      save_individual= True
      
    )
    try:
      save_demos(rets["random_demos"], f"{seed}-random_demos")
    except:
      print("Error saving demos, dont really care")


In [ ]:
seeds = np.random.randint(0, high = 10000, size = (5))
# ## uusing the same seeds as the first half, crashed run

# # seeds = [
# #   1255, 3370, 3486, 3944, 8614, 
# # ]

# for param in [ training_params1000ep, training_params2000ep ]:

## this is found to be the one that works the best?
param = training_params400ep

for seed in seeds:
    name = f"proprio-check--2*1-attn-rnncomp{param['epochs']}ep-10mbsize-{seed}loaderseed-001lr"

    param["lock_loader_seed"] = seed

    agents = [    
      agent_rnn_wrist_rgb_attn(),
      agent_rnn_wrist_rgb_attn_proprio(),
    ]

    params = [param] * len(agents)

    frame, rets = train_and_test_vision_config(
      env, 
      agents,
      params, 
      normal_demos,
      smaller_demos,
      task, 
      normal_size_task_params, 
      smaller_task_params, 
      df_csv_name=name,
      check_random_demos=10,
      save_individual= True
      
    )
    try:
      save_demos(rets["random_demos"], f"{seed}-random_demos")
    except:
      print("Error saving demos, dont really care")


In [ ]:

ag1 = lambda: Agent(
  env.action_shape[0],
  policy_type=PolicyType.RNN_GRASP,
  cam_type=CamType.WRIST_DEPTH | CamType.RIGHT_SHOULDER, 
  rnn_opts = {
    "config": "depth_feats",
    "use_proprio": False
  }
)

ag1_proprio = lambda: Agent(
  env.action_shape[0],
  policy_type=PolicyType.RNN_GRASP,
  cam_type= CamType.WRIST_DEPTH | CamType.RIGHT_SHOULDER, 
  rnn_opts = {
    "config": "depth_feats",
    "use_proprio": True
  }
)

ag2_attn = lambda: Agent(
  env.action_shape[0],
  policy_type=PolicyType.RNN_GRASP,
  cam_type= CamType.WRIST_DEPTH | CamType.RIGHT_SHOULDER, 
  rnn_opts = {
    "config": "attn",
    "attn_opts": {"is_deep_fuse": True},
    "use_proprio": False
  }
)

ag2_attn_proprio = lambda: Agent(
  env.action_shape[0],
  policy_type=PolicyType.RNN_GRASP,
  cam_type= CamType.WRIST_DEPTH | CamType.RIGHT_SHOULDER, 
  rnn_opts = {
    "config": "attn",
    "attn_opts": {"is_deep_fuse": True},
    "use_proprio": True
  }
)


In [ ]:
seeds = np.random.randint(0, high = 10000, size = (5))
# ## uusing the same seeds as the first half, crashed run


## this is found to be the one that works the best?
param = training_params400ep

for seed in seeds:
    name = f"r_shoulder_wd--proprio-check--2*1-attn-rnncomp{param['epochs']}ep-10mbsize-{seed}loaderseed-001lr"

    param["lock_loader_seed"] = seed

    agents = [    
      ag1(),
      ag1_proprio(),
      ag2_attn(),
      ag2_attn_proprio(),
    ]

    params = [param] * len(agents)

    frame, rets = train_and_test_vision_config(
      env, 
      agents,
      params, 
      normal_demos,
      smaller_demos,
      task, 
      normal_size_task_params, 
      smaller_task_params, 
      df_csv_name=name,
      check_random_demos=10,
      save_individual= True
      
    )
    try:
      save_demos(rets["random_demos"], f"{seed}-random_demos")
    except:
      print("Error saving demos, dont really care")


### Depth Grasp Proprioceptive
including it here as well


In [5]:
task = Vision_Random

## Train on 1 to 1 scale and fixed distance
normal_size_task_params = {
  "scale" : 1.0, 
  "wrist_cam_distance": 0.6
}

env._dataset_root = "data/demos20-normal-Vision_Random"
task_env = env.get_task(task, **normal_size_task_params)
normal_demos = task_env.get_demos(10, live_demos=False)


## test on half the distance but smaller
smaller_task_params = {
  "scale" : 0.5, 
  "wrist_cam_distance": 0.3
}
env._dataset_root = "data/demos20-small-Vision_Random"
task_env = env.get_task(task, **smaller_task_params)
smaller_demos = task_env.get_demos(10, live_demos=False)


In [6]:
training_params2000ep = {
  "epochs": 2000,
  "minibatch_size": 10,
  "shuffle_data": True,
  "lock_loader_seed": None, ## the loop will choose this
  "lr": 1e-3,
}

In [7]:

agent_rgb = lambda: Agent(
  env.action_shape[0],
  policy_type=PolicyType.DEPTH_GRASP,
  cam_type=CamType.WRIST, 
  config = "depth_ch"
)
agent_wd = lambda: Agent(
  env.action_shape[0],
  policy_type=PolicyType.DEPTH_GRASP,
  cam_type=CamType.WRIST | CamType.WRIST_DEPTH, 
  config = "depth_ch"
)

agent_depth_feats_gate = lambda: Agent(
  env.action_shape[0],
  policy_type=PolicyType.DEPTH_GRASP,
  cam_type=CamType.WRIST | CamType.WRIST_DEPTH, 
  config = "depth_feats",
  opts = {"gated_fuse": True}
)

agent_depth_feats_gate_l_r = lambda: Agent(
  env.action_shape[0],
  policy_type=PolicyType.DEPTH_GRASP,
  cam_type=CamType.LEFT_SHOULDER | CamType.RIGHT_SHOULDER | CamType.WRIST_DEPTH, 
  config = "depth_feats",
  opts = {"gated_fuse": True}
)

agent_rgb_proprio = lambda: Agent(
  env.action_shape[0],
  policy_type=PolicyType.DEPTH_GRASP,
  cam_type=CamType.WRIST, 
  config = "depth_ch",
  opts = {"use_proprio": True}
)
agent_wd_proprio = lambda: Agent(
  env.action_shape[0],
  policy_type=PolicyType.DEPTH_GRASP,
  cam_type=CamType.WRIST | CamType.WRIST_DEPTH, 
  config = "depth_ch",
  opts = {
    "use_proprio": True
  }
)

agent_depth_feats_gate_proprio = lambda: Agent(
  env.action_shape[0],
  policy_type=PolicyType.DEPTH_GRASP,
  cam_type=CamType.WRIST | CamType.WRIST_DEPTH, 
  config = "depth_feats",
  opts = {"gated_fuse": True,
          "use_proprio": True
        }
)

agent_depth_feats_gate_l_r_proprio = lambda: Agent(
  env.action_shape[0],
  policy_type=PolicyType.DEPTH_GRASP,
  cam_type=CamType.LEFT_SHOULDER | CamType.RIGHT_SHOULDER | CamType.WRIST_DEPTH, 
  config = "depth_feats",
  opts = {"gated_fuse": True,
          "use_proprio": True}
)


agent_rgb() 
agent_wd()
agent_depth_feats_gate()
agent_depth_feats_gate_l_r()

agent_rgb_proprio() 
agent_wd_proprio()
agent_depth_feats_gate_proprio()
agent_depth_feats_gate_l_r_proprio()



[simple_policy] - Policy] Using wrist as camera type
[simple_policy] - Policy] Using wrist+wrist_depth as camera type
[simple_policy] - Policy] Using wrist+wrist_depth as camera type
[simple_policy] - Policy] Using l_shoulder+r_shoulder+wrist_depth as camera type
[simple_policy] - Policy] Using wrist as camera type
[simple_policy] - Policy] Using wrist+wrist_depth as camera type
[simple_policy] - Policy] Using wrist+wrist_depth as camera type
[simple_policy] - Policy] Using l_shoulder+r_shoulder+wrist_depth as camera type


Agent(policy_type=depth_grasp_policy, cam_type=l_shoulder+r_shoulder+wrist_depth, policy=depth_grasp_policy-config:depth_feats-opts:{'gated_fuse': True, 'attn_num_heads': 8, 'attn_deep_fuse': True, 'use_proprio': True, 'proprio_opts': {}})

In [8]:
seeds = np.random.randint(0, high = 10000, size = (10))

## TODO: rerun for 8130
seeds = [
 580, 1679, 2744, 3147, 3506, 4249, 7004, 7253, 8130, 9435
]
params = training_params2000ep

for seed in seeds:
  
  name = f"wrist-proprio--proprio-graspcomp-8way-{params['epochs']}ep-10mbsize-{seed}loaderseed-001lr"
  params["lock_loader_seed"] = seed

  agents = [
    # agent_rgb(),
    agent_rgb_proprio() ,
    # agent_wd(),
    # agent_wd_proprio(),
    # agent_depth_feats_gate(),
    # agent_depth_feats_gate_proprio(),
    # agent_depth_feats_gate_l_r(),
    # agent_depth_feats_gate_l_r_proprio(),
  ]

  ag_params = [params] * len(agents)

  frame, rets = train_and_test_vision_config(
    env, 
    agents,
    ag_params, 
    normal_demos,
    smaller_demos,
    task, 
    normal_size_task_params, 
    smaller_task_params, 
    df_csv_name=name,
    check_random_demos=10,
    save_individual= True
  )
  save_demos(rets["random_demos"], f"{seed}-random_demos")

seeds

[simple_policy] - Policy] Using wrist as camera type
Training Agent 'agent-policy:depth_grasp_policy-cams:wrist-policy:depth_grasp_policy-config:depth_ch-opts:{'gated_fuse': True, 'attn_num_heads': 8, 'attn_deep_fuse': True, 'use_proprio': True, 'proprio_opts': {}}'
2- Using given demos for task: Vision_Random
Training params: 
	epochs = 2000,
	model_path = None,
	shuffle_data = True,
	minibatch_size = 10,
	lr = 0.001,
	data_label = joint_velocities,
	shuffle_obs_in_demo = False [not being used!],
	lambda_grasp_loss = 1,
	dataset_to_use = demo [not being used!],
	lock_loader_seed = 580,
	device = cuda,
	


100%|██████████| 2000/2000 [04:18<00:00,  7.72it/s]


Done Training Policy on 20 Demos

Testing agents on the trained demos (train_demos):
[vision_random] scale: 1.0 (picked at creation)
[vision_random] dist: 0.6 (picked at creation)
[vision_random] scale: 1.0 (picked at creation)
[vision_random] dist: 0.6 (picked at creation)
[vision_random] scale: 1.0 (picked at creation)
[vision_random] dist: 0.6 (picked at creation)
[vision_random] scale: 1.0 (picked at creation)
[vision_random] dist: 0.6 (picked at creation)
[vision_random] scale: 1.0 (picked at creation)
[vision_random] dist: 0.6 (picked at creation)
[vision_random] scale: 1.0 (picked at creation)
[vision_random] dist: 0.6 (picked at creation)
[vision_random] scale: 1.0 (picked at creation)
[vision_random] dist: 0.6 (picked at creation)
[vision_random] scale: 1.0 (picked at creation)
[vision_random] dist: 0.6 (picked at creation)
[vision_random] scale: 1.0 (picked at creation)
[vision_random] dist: 0.6 (picked at creation)
[vision_random] scale: 1.0 (picked at creation)
[vision_rand

100%|██████████| 2000/2000 [04:06<00:00,  8.11it/s]


Done Training Policy on 20 Demos

Testing agents on the trained demos (train_demos):
[vision_random] scale: 1.0 (picked at creation)
[vision_random] dist: 0.6 (picked at creation)
[vision_random] scale: 1.0 (picked at creation)
[vision_random] dist: 0.6 (picked at creation)
[vision_random] scale: 1.0 (picked at creation)
[vision_random] dist: 0.6 (picked at creation)
[vision_random] scale: 1.0 (picked at creation)
[vision_random] dist: 0.6 (picked at creation)
[vision_random] scale: 1.0 (picked at creation)
[vision_random] dist: 0.6 (picked at creation)
[vision_random] scale: 1.0 (picked at creation)
[vision_random] dist: 0.6 (picked at creation)
[vision_random] scale: 1.0 (picked at creation)
[vision_random] dist: 0.6 (picked at creation)
[vision_random] scale: 1.0 (picked at creation)
[vision_random] dist: 0.6 (picked at creation)
[vision_random] scale: 1.0 (picked at creation)
[vision_random] dist: 0.6 (picked at creation)
[vision_random] scale: 1.0 (picked at creation)
[vision_rand

...

100%|██████████| 2000/2000 [04:23<00:00,  7.58it/s]


Done Training Policy on 20 Demos

Testing agents on the trained demos (train_demos):
[vision_random] scale: 1.0 (picked at creation)
[vision_random] dist: 0.6 (picked at creation)
[vision_random] scale: 1.0 (picked at creation)
[vision_random] dist: 0.6 (picked at creation)
[vision_random] scale: 1.0 (picked at creation)
[vision_random] dist: 0.6 (picked at creation)
[vision_random] scale: 1.0 (picked at creation)
[vision_random] dist: 0.6 (picked at creation)
[vision_random] scale: 1.0 (picked at creation)
[vision_random] dist: 0.6 (picked at creation)
[vision_random] scale: 1.0 (picked at creation)
[vision_random] dist: 0.6 (picked at creation)
[vision_random] scale: 1.0 (picked at creation)
[vision_random] dist: 0.6 (picked at creation)
[vision_random] scale: 1.0 (picked at creation)
[vision_random] dist: 0.6 (picked at creation)
[vision_random] scale: 1.0 (picked at creation)
[vision_random] dist: 0.6 (picked at creation)
[vision_random] scale: 1.0 (picked at creation)
[vision_rand

100%|██████████| 2000/2000 [04:12<00:00,  7.91it/s]


Done Training Policy on 20 Demos

Testing agents on the trained demos (train_demos):
[vision_random] scale: 1.0 (picked at creation)
[vision_random] dist: 0.6 (picked at creation)
[vision_random] scale: 1.0 (picked at creation)
[vision_random] dist: 0.6 (picked at creation)
[vision_random] scale: 1.0 (picked at creation)
[vision_random] dist: 0.6 (picked at creation)
[vision_random] scale: 1.0 (picked at creation)
[vision_random] dist: 0.6 (picked at creation)
[vision_random] scale: 1.0 (picked at creation)
[vision_random] dist: 0.6 (picked at creation)
[vision_random] scale: 1.0 (picked at creation)
[vision_random] dist: 0.6 (picked at creation)
[vision_random] scale: 1.0 (picked at creation)
[vision_random] dist: 0.6 (picked at creation)
[vision_random] scale: 1.0 (picked at creation)
[vision_random] dist: 0.6 (picked at creation)
[vision_random] scale: 1.0 (picked at creation)
[vision_random] dist: 0.6 (picked at creation)
[vision_random] scale: 1.0 (picked at creation)
[vision_rand

[580, 1679, 2744, 3147, 3506, 4249, 7004, 7253, 8130, 9435]

### now with the `attn` thing


In [ ]:
agent_attn_w_wd = lambda: Agent(
  env.action_shape[0],
  policy_type=PolicyType.DEPTH_GRASP,
  cam_type=CamType.WRIST | CamType.WRIST_DEPTH, 
  grasp_thresh = 0.5,
  config = "attn",
  opts = {
  }
)

agent_attn_deep_w_wd = lambda: Agent(
  env.action_shape[0],
  policy_type=PolicyType.DEPTH_GRASP,
  cam_type=CamType.WRIST | CamType.WRIST_DEPTH, 
  grasp_thresh = 0.5,
  config = "attn",
    opts = {
    "attn_deep_fuse": True,
  }
)
agent_attn_deep_l_r_wd = lambda: Agent(
  env.action_shape[0],
  policy_type=PolicyType.DEPTH_GRASP,
  cam_type=CamType.LEFT_SHOULDER | CamType.RIGHT_SHOULDER | CamType.WRIST_DEPTH, 
  grasp_thresh = 0.5,
  config = "attn",
    opts = {
    "attn_deep_fuse": True,
  }
)

agent_attn_l_r_wd = lambda: Agent(
  env.action_shape[0],
  policy_type=PolicyType.DEPTH_GRASP,
  cam_type=CamType.LEFT_SHOULDER | CamType.RIGHT_SHOULDER | CamType.WRIST_DEPTH, 
  grasp_thresh = 0.5,
  config = "attn",
    opts = {
    "attn_deep_fuse": False,
  }
)

agent_attn_w_wd_proprio = lambda: Agent(
  env.action_shape[0],
  policy_type=PolicyType.DEPTH_GRASP,
  cam_type=CamType.WRIST | CamType.WRIST_DEPTH, 
  grasp_thresh = 0.5,
  config = "attn",
  opts = {
    "use_proprio": True
  }
)

agent_attn_deep_w_wd_proprio = lambda: Agent(
  env.action_shape[0],
  policy_type=PolicyType.DEPTH_GRASP,
  cam_type=CamType.WRIST | CamType.WRIST_DEPTH, 
  grasp_thresh = 0.5,
  config = "attn",
    opts = {
    "attn_deep_fuse": True,
    "use_proprio": True
  }
)
agent_attn_deep_l_r_wd_proprio = lambda: Agent(
  env.action_shape[0],
  policy_type=PolicyType.DEPTH_GRASP,
  cam_type=CamType.LEFT_SHOULDER | CamType.RIGHT_SHOULDER | CamType.WRIST_DEPTH, 
  grasp_thresh = 0.5,
  config = "attn",
    opts = {
    "attn_deep_fuse": True,
    "use_proprio": True
  }
)

agent_attn_l_r_wd_proprio = lambda: Agent(
  env.action_shape[0],
  policy_type=PolicyType.DEPTH_GRASP,
  cam_type=CamType.LEFT_SHOULDER | CamType.RIGHT_SHOULDER | CamType.WRIST_DEPTH, 
  grasp_thresh = 0.5,
  config = "attn",
    opts = {
    "attn_deep_fuse": False,
    "use_proprio": True
  }
)

agent_attn_w_wd()
agent_attn_deep_w_wd()
agent_attn_deep_l_r_wd()
agent_attn_l_r_wd()

agent_attn_w_wd_proprio()
agent_attn_deep_w_wd_proprio()
agent_attn_deep_l_r_wd_proprio()
agent_attn_l_r_wd_proprio()


In [ ]:
seeds = np.random.randint(0, high = 10000, size = (10))

params = training_params2000ep

for seed in seeds:
  
  name = f"proprio-attngrasp-8way-{params['epochs']}ep-10mbsize-{seed}loaderseed-001lr"
  params["lock_loader_seed"] = seed

  agents = [
    agent_rgb(),
    # agent_rgb_proprio() ,
    agent_wd(),
    agent_wd_proprio(),
    agent_depth_feats_gate(),
    agent_depth_feats_gate_proprio(),
    agent_depth_feats_gate_l_r(),
    agent_depth_feats_gate_l_r_proprio(),
  ]

  ag_params = [params] * len(agents)

  frame, rets = train_and_test_vision_config(
    env, 
    agents,
    ag_params, 
    normal_demos,
    smaller_demos,
    task, 
    normal_size_task_params, 
    smaller_task_params, 
    df_csv_name=name,
    check_random_demos=10,
    save_individual= True
  )
  save_demos(rets["random_demos"], f"{seed}-random_demos")

seeds